Before diving into the core implementation of the biometric engine, let's interactively visualize the two mathematical pillars of modern speaker recognition: **Cosine Similarity** and **Latent Space Clustering**.

In [1]:
# ENVIRONMENT SETUP & DEPENDENCIES
%pip install -v -i https://package-mirror.liara.ir/repository/pypi/simple numpy matplotlib scipy ipywidgets scikit-learn --trusted-host package-mirror.liara.ir --quiet

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.patches import Circle
from numpy.linalg import norm
import warnings

from IPython.display import display, clear_output
from ipywidgets import interact, FloatSlider, Button, VBox, HBox, Output
import ipywidgets as widgets

warnings.filterwarnings('ignore')
plt.style.use('dark_background')

print("✅ Laboratory Environment Initialized.")

Looking in indexes: https://package-mirror.liara.ir/repository/pypi/simple
Note: you may need to restart the kernel to use updated packages.
✅ Laboratory Environment Initialized.


### 1. The Geometry of Verification (Cosine Similarity)
Adjust the angles of the vectors below to simulate how the neural network compares a **Stored Identity Vector** with a **Live Audio Vector**. Notice how the angle directly controls the authentication status.

In [2]:
# INTERACTIVE COSINE SIMILARITY WIDGET
@interact
def interactive_cosine(
    angle_a = FloatSlider(value=45, min=0, max=360, step=5, description='Vector A Angle:'),
    angle_b = FloatSlider(value=60, min=0, max=360, step=5, description='Vector B Angle:')
):
    rad_a, rad_b = np.radians(angle_a), np.radians(angle_b)
    v1 = np.array([np.cos(rad_a), np.sin(rad_a)])
    v2 = np.array([np.cos(rad_b), np.sin(rad_b)])
    
    cos_sim = np.dot(v1, v2) / (norm(v1) * norm(v2))
    
    threshold = 0.85
    is_match = cos_sim >= threshold
    status_color = "#00ffcc" if is_match else "#ff0055"
    status_text = "ACCESS GRANTED" if is_match else "ACCESS DENIED"
    
    fig, ax = plt.subplots(figsize=(8, 8))
    ax.set_xlim(-1.5, 1.5)
    ax.set_ylim(-1.5, 1.5)
    
    circle = Circle((0, 0), 1, fill=False, color='#333344', linestyle='--')
    ax.add_patch(circle)
    
    ax.quiver(0, 0, v1[0], v1[1], angles='xy', scale_units='xy', scale=1, color='#00d4ff', width=0.015, label='Enrolled Voice (Database)')
    ax.quiver(0, 0, v2[0], v2[1], angles='xy', scale_units='xy', scale=1, color='#ff9800', width=0.015, label='Live Audio (Microphone)')
    
    ax.axhline(0, color='#333344', linewidth=1)
    ax.axvline(0, color='#333344', linewidth=1)
    
    ax.set_facecolor('#0f0f1a')
    fig.patch.set_facecolor('#0f0f1a')
    plt.title(f"Cosine Similarity: {cos_sim:.3f}\n[{status_text}]", color=status_color, fontsize=16, fontweight='bold', pad=20)
    plt.legend(loc='upper right', facecolor='#1a1a2e', edgecolor='none', labelcolor='white')
    plt.grid(color='#ffffff', alpha=0.05)
    plt.show()

interactive(children=(FloatSlider(value=45.0, description='Vector A Angle:', max=360.0, step=5.0), FloatSlider…

### 2. Navigating the Latent Space
Deep Neural Networks map distinct voices into separate clusters in a multi-dimensional space. Click the button to simulate a live audio capture and see if the resulting vector falls within your established security boundary.

In [ ]:
# LATENT SPACE SIMULATION WIDGET
clusters = {
    "Target User (You)": np.array([2.0, 3.0]),
    "Imposter 1": np.array([-3.0, 1.5]),
    "Environmental Noise": np.array([0.5, -3.0])
}

out_plot = Output()
simulate_btn = Button(description='🎙️ Simulate Live Audio Capture', button_style='primary')

def plot_embedding_space(live_vector=None):
    with out_plot:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(10, 6))
        
        colors = ['#00ffcc', '#ff9800', '#ff0055']
        for (name, center), color in zip(clusters.items(), colors):
            points = np.random.randn(50, 2) * 0.4 + center
            ax.scatter(points[:, 0], points[:, 1], c=color, alpha=0.3, edgecolors='none', s=30)
            ax.scatter(center[0], center[1], c=color, marker='X', s=200, edgecolors='white', label=f"{name} (Centroid)")
            
            if name == "Target User (You)":
                circle = Circle((center[0], center[1]), 1.2, fill=False, color=color, linestyle='--', alpha=0.8)
                ax.add_patch(circle)
                ax.text(center[0]-1.3, center[1]+1.3, "Security Boundary (Threshold)", color=color, fontsize=9)

        if live_vector is not None:
            dist = norm(live_vector - clusters["Target User (You)"])
            is_auth = dist <= 1.2
            res_color = "#00ffcc" if is_auth else "#ff0055"
            ax.scatter(live_vector[0], live_vector[1], c=res_color, marker='*', s=400, edgecolors='white', zorder=5)
            ax.annotate('LIVE AUDIO', (live_vector[0], live_vector[1]+0.3), color=res_color, fontweight='bold', ha='center')
            
            status = "GRANTED" if is_auth else "DENIED"
            plt.title(f"Vector Distance to Target: {dist:.2f} | Status: ACCESS {status}", color=res_color, fontsize=14, fontweight='bold', pad=15)
        else:
            plt.title("Latent Space Mapping (D-Vectors/X-Vectors)", color='white', fontsize=14, pad=15)

        ax.set_facecolor('#0f0f1a')
        fig.patch.set_facecolor('#0f0f1a')
        plt.legend(loc='lower left', facecolor='#1a1a2e', edgecolor='none', labelcolor='white')
        plt.grid(color='#ffffff', alpha=0.05)
        plt.xlim(-5, 5)
        plt.ylim(-5, 5)
        plt.show()

def on_simulate_click(b):
    if np.random.rand() > 0.6:
        live_vec = clusters["Target User (You)"] + (np.random.randn(2) * 0.5)
    else:
        live_vec = (np.random.rand(2) * 8) - 4
    plot_embedding_space(live_vector=live_vec)

simulate_btn.on_click(on_simulate_click)
display(VBox([simulate_btn, out_plot]))
plot_embedding_space()

**Chapter 4: The Edge of Science and Global Trends**

**Lesson 11:**



# 🧩 Project 5: Biometric Voice Authentication (Speaker Recognition)


### 1️⃣ Installing and Importing Libraries


In [ ]:
%pip install -v -i https://package-mirror.liara.ir/repository/pypi/simple pyaudio librosa matplotlib scikit-learn --trusted-host package-mirror.liara.ir --quiet

In [4]:
import numpy as np
import librosa
import pyaudio
import time
import os
import pickle
import warnings
from sklearn.mixture import GaussianMixture
from IPython.display import Audio, display, HTML, clear_output
import matplotlib.pyplot as plt

warnings.filterwarnings('ignore')
print("✅ Libraries loaded successfully.")

✅ Libraries loaded successfully.


### 2️⃣ Helper Functions: Recording & Feature Extraction



We first define a simple recorder that grabs audio from your microphone and a feature extractor based on **MFCC + deltas** (39‑dimensional vector per frame).

In [5]:
RATE = 16000
CHUNK = 1024
FORMAT = pyaudio.paInt16
CHANNELS = 1

def record_voice(duration=3, prompt="Speak now..."):
    p = pyaudio.PyAudio()
    stream = p.open(format=FORMAT, channels=CHANNELS, rate=RATE,
                    input=True, frames_per_buffer=CHUNK)
    print(f"🎤 {prompt}")
    frames = []
    for _ in range(0, int(RATE / CHUNK * duration)):
        data = stream.read(CHUNK, exception_on_overflow=False)
        frames.append(np.frombuffer(data, dtype=np.int16))
    stream.stop_stream()
    stream.close()
    p.terminate()
    audio = np.concatenate(frames).astype(np.float32) / 32768.0
    return audio

def extract_speaker_features(wav, sr=16000, n_mfcc=20):
    intervals = librosa.effects.split(wav, top_db=30)
    wav_voiced = np.concatenate([wav[start:end] for start, end in intervals])
    if len(wav_voiced) < sr * 0.1: 
        wav_voiced = wav
    mfcc = librosa.feature.mfcc(y=wav_voiced, sr=sr, n_mfcc=n_mfcc, n_fft=512, hop_length=160)
    mfcc = mfcc[1:, :]
    mfcc = mfcc - np.mean(mfcc, axis=1, keepdims=True)
    return mfcc.T

def score_utterance(wav, gmm, min_rms=0.005):  
    rms = np.sqrt(np.mean(wav**2))
    if rms < min_rms:
        return -np.inf      
    feats = extract_speaker_features(wav)
    if len(feats) < 5:     
        return -np.inf
    return gmm.score(feats)

### 3️⃣ Speaker Enrollment


The user provides **3 distinct natural sentences** (no fixed passphrase). For each sample we extract features, concatenate them, and train an 8‑component diagonal‑covariance GMM.


In [6]:

def enroll_speaker(name, duration_per_sample=4, num_samples=3):
    print(f"\n📝 Enrolling speaker: {name}")
    print("🎤 Please say a completely different, natural sentence each time (not a fixed phrase).")
    all_features = []
    for i in range(num_samples):
        print(f"\n  Sample {i+1} of {num_samples}:")
        wav = record_voice(duration=duration_per_sample, prompt="Speak (e.g. tell about your day)")
        wav, _ = librosa.effects.trim(wav, top_db=20)
        feats = extract_speaker_features(wav)
        all_features.append(feats)
        time.sleep(0.5)
    X = np.vstack(all_features)
    gmm = GaussianMixture(n_components=8, covariance_type='diag',
                          max_iter=200, random_state=42, reg_covar=1e-3)
    gmm.fit(X)
    # Compute average enrolment score for calibration
    scores = [gmm.score(f) for f in all_features]
    avg_score = np.mean(scores)
    print(f"✅ Model for {name} built. Average enrolment score: {avg_score:.2f}\n")
    return gmm, avg_score

# Enroll the primary user
speaker_model, enroll_avg = enroll_speaker("Main User", duration_per_sample=4, num_samples=3)


📝 Enrolling speaker: Main User
🎤 Please say a completely different, natural sentence each time (not a fixed phrase).

  Sample 1 of 3:
🎤 Speak (e.g. tell about your day)

  Sample 2 of 3:
🎤 Speak (e.g. tell about your day)

  Sample 3 of 3:
🎤 Speak (e.g. tell about your day)
✅ Model for Main User built. Average enrolment score: -71.60



### 4️⃣ Threshold Calibration



We collect **two more utterances**: one from the enrolled speaker (a **new** sentence) and one from a different person (optional). The threshold is set halfway between the enrolment score and the impostor score.

In [9]:

def calibrate_threshold(gmm, enroll_score, n_other=2):
    print("⚖️ Threshold calibration:")
    print("   1st: say a new sentence (different from enrolment).")
    print("   2nd: a friend or family member speaks (optional).")
    scores = []
    for i in range(n_other):
        prompt = "New sentence from yourself" if i == 0 else "Another person speaks"
        wav = record_voice(duration=3, prompt=prompt)
        wav, _ = librosa.effects.trim(wav, top_db=20)
        feats = extract_speaker_features(wav)
        if len(feats) > 5:
            s = gmm.score(feats) 
            scores.append(s)
            print(f"   Score: {s:.2f}")
        else:
            print("   ⚠️ Too short, ignored.")
    if scores:
        avg_other = np.mean(scores)
        threshold = (enroll_score + avg_other) / 2
        print(f"\n✅ Threshold: {threshold:.2f} (enrol: {enroll_score:.2f}, other: {avg_other:.2f})")
    else:
        threshold = enroll_score - 5
        print(f"⚠️ Fallback threshold: {threshold:.2f}")
    return threshold

THRESHOLD = calibrate_threshold(speaker_model, enroll_avg, n_other=2)

⚖️ Threshold calibration:
   1st: say a new sentence (different from enrolment).
   2nd: a friend or family member speaks (optional).
🎤 New sentence from yourself
   Score: -74.58
🎤 Another person speaks
   Score: -79.92

✅ Threshold: -74.42 (enrol: -71.60, other: -77.25)


### 5️⃣ Futuristic Verification UI

This function renders a cyber‑styled HUD with a confidence gauge, using pure HTML/CSS and SVG. It accepts the mode (`'verify'` or `'identify'`) and the relevant data.


In [ ]:
from IPython.display import display, HTML, clear_output
import time

def futuristic_speaker_ui(mode, status, score=None, threshold=None):
    clear_output(wait=True)

    # Base styles (same as before)
    base_styles = """
    <style>
        @import url('https://fonts.googleapis.com/css2?family=Orbitron:wght@500;700;900&display=swap');

        .hud-container {
            position: relative;
            background: rgba(10, 10, 18, 0.85);
            backdrop-filter: blur(15px);
            -webkit-backdrop-filter: blur(15px);
            border-radius: 20px;
            padding: 35px;
            max-width: 450px;
            margin: 20px auto;
            font-family: 'Orbitron', 'Courier New', monospace;
            text-align: center;
            overflow: hidden;
            direction: ltr;
        }

        .scanline {
            width: 100%; height: 3px; position: absolute; top: 0; left: 0; opacity: 0.6;
            animation: scan 2.5s linear infinite;
        }

        @keyframes scan {
            0% { top: -10%; }
            100% { top: 110%; }
        }

        .digital-number { font-family: 'Orbitron', sans-serif; letter-spacing: 2px; }

        @keyframes pulse-glow {
            0% { filter: drop-shadow(0 0 10px currentColor); }
            50% { filter: drop-shadow(0 0 30px currentColor); }
            100% { filter: drop-shadow(0 0 10px currentColor); }
        }
    </style>
    """

    if mode == 'verify':
        is_accepted = (status == 'unlocked')
        color_main = "#00ffcc" if is_accepted else "#ff0055"
        bg_glow = "rgba(0, 255, 204, 0.15)" if is_accepted else "rgba(255, 0, 85, 0.15)"
        title_text = "ACCESS GRANTED" if is_accepted else "ACCESS DENIED"
        status_text = "Identity verified successfully" if is_accepted else "Voice mismatch. Lockdown active"

        # Simple icon – shield (accept) or lock (deny)
        if is_accepted:
            icon_svg = f"""
            <svg viewBox="0 0 24 24" width="100" height="100" stroke="{color_main}" stroke-width="1.5" fill="none" stroke-linecap="round" stroke-linejoin="round" style="animation: pulse-glow 2s infinite; color: {color_main}; margin-bottom: 20px;">
                <path d="M12 22s8-4 8-10V5l-8-3-8 3v7c0 6 8 10 8 10z"/>
                <path d="M9 12l2 2 4-4" stroke-dasharray="20" stroke-dashoffset="20">
                    <animate attributeName="stroke-dashoffset" values="20;0" dur="0.8s" fill="freeze" />
                </path>
            </svg>
            """
        else:
            icon_svg = f"""
            <svg viewBox="0 0 24 24" width="100" height="100" stroke="{color_main}" stroke-width="1.5" fill="none" stroke-linecap="round" stroke-linejoin="round" style="animation: pulse-glow 0.5s infinite; color: {color_main}; margin-bottom: 20px;">
                <rect x="3" y="11" width="18" height="11" rx="2" ry="2"/>
                <path d="M7 11V7a5 5 0 0 1 10 0v4"/>
                <line x1="12" y1="15" x2="12" y2="17" stroke-width="3"/>
            </svg>
            """

        html = base_styles + f"""
        <style>
            .hud-verify {{
                border: 1px solid rgba(255,255,255,0.05);
                border-bottom: 3px solid {color_main};
                box-shadow: 0 15px 35px {bg_glow}, inset 0 0 20px {bg_glow};
            }}
            .scanline-verify {{ background: {color_main}; box-shadow: 0 0 15px {color_main}; }}
        </style>

        <div class="hud-container hud-verify">
            <div class="scanline scanline-verify"></div>
            {icon_svg}
            <h1 style="color: #fff; margin: 0 0 10px 0; font-size: 32px; text-shadow: 0 0 20px {color_main}; letter-spacing: 2px;">
                {title_text}
            </h1>
            <div style="color: {color_main}; font-size: 14px; letter-spacing: 1px; margin-top: 15px; border-top: 1px solid rgba(255,255,255,0.1); padding-top: 15px;">
                {status_text}
            </div>
        </div>
        """

    else:  # 'identify' mode – absolutely unchanged
        speaker_name = status if status else "Unknown Entity"
        color_main = "#00d4ff"

        # Radar / AI core SVG
        icon_svg = f"""
        <div style="position: relative; width: 100px; height: 100px; margin: 0 auto 15px auto;">
            <svg viewBox="0 0 100 100" style="position: absolute; top: 0; left: 0; width: 100%; height: 100%; animation: spin-slow 8s linear infinite;">
                <circle cx="50" cy="50" r="45" fill="none" stroke="{color_main}" stroke-width="1" stroke-dasharray="10 5 30 10" opacity="0.5"/>
                <circle cx="50" cy="50" r="35" fill="none" stroke="{color_main}" stroke-width="2" stroke-dasharray="40 20" opacity="0.8"/>
            </svg>
            <svg viewBox="0 0 24 24" width="46" height="46" stroke="{color_main}" stroke-width="1.2" fill="none" stroke-linecap="round" stroke-linejoin="round" style="position: absolute; top: 27px; left: 27px; filter: drop-shadow(0 0 10px {color_main});">
                <path d="M20 21v-2a4 4 0 0 0-4-4H8a4 4 0 0 0-4 4v2"/>
                <circle cx="12" cy="7" r="4"/>
            </svg>
        </div>
        """

        html = base_styles + f"""
        <style>
            .hud-identify {{
                border: 1px solid rgba(0, 212, 255, 0.15);
                border-top: 3px solid {color_main};
                box-shadow: 0 15px 40px rgba(0, 212, 255, 0.1), inset 0 0 30px rgba(0, 212, 255, 0.05);
            }}
            .scanline-identify {{ background: {color_main}; box-shadow: 0 0 15px {color_main}; }}

            .audio-waves {{ display: flex; justify-content: center; align-items: center; gap: 5px; height: 50px; margin: 25px 0; }}
            .wave-bar {{
                width: 4px; background: {color_main}; border-radius: 2px;
                box-shadow: 0 0 10px {color_main};
                animation: wave-bounce 1s ease-in-out infinite alternate;
            }}
            .wave-bar:nth-child(1) {{ height: 15px; animation-delay: 0.1s; }}
            .wave-bar:nth-child(2) {{ height: 30px; animation-delay: 0.3s; }}
            .wave-bar:nth-child(3) {{ height: 50px; animation-delay: 0.0s; }}
            .wave-bar:nth-child(4) {{ height: 35px; animation-delay: 0.4s; }}
            .wave-bar:nth-child(5) {{ height: 20px; animation-delay: 0.2s; }}

            @keyframes wave-bounce {{
                0% {{ transform: scaleY(0.2); opacity: 0.4; }}
                100% {{ transform: scaleY(1); opacity: 1; }}
            }}
        </style>

        <div class="hud-container hud-identify">
            <div class="scanline scanline-identify"></div>
            {icon_svg}
            <div style="color: #666; font-size: 11px; letter-spacing: 3px; margin-bottom: 5px;" class="digital-number">IDENTIFIED ENTITY</div>
            <h1 style="color: #fff; margin: 0; font-size: 32px; text-shadow: 0 0 20px {color_main};">
                {speaker_name}
            </h1>
            <div class="audio-waves">
                <div class="wave-bar"></div><div class="wave-bar"></div><div class="wave-bar"></div>
                <div class="wave-bar"></div><div class="wave-bar"></div>
            </div>
            <div class="digital-number" style="color: {color_main}; font-size: 11px; opacity: 0.7; letter-spacing: 1px; border-top: 1px dashed rgba(0,212,255,0.2); padding-top: 15px;">
                NEURAL NETWORK LISTENING...
            </div>
        </div>
        """


    display(HTML(html))
    time.sleep(0.5)

### 6️⃣ Live Speaker Verification


Now we put everything together: the system continuously listens, computes the utterance score, compares it with `THRESHOLD`, and shows the result on the HUD.


In [ ]:
import librosa
import time

print("🎙️ Biometric verification system activated. Please speak. Ctrl+C to stop.\n")
time.sleep(1)

try:
    while True:
        test_wav = record_voice(duration=3, prompt="Speak to verify...")
        test_wav, _ = librosa.effects.trim(test_wav, top_db=20)
        
        if len(test_wav) < RATE * 0.5:
            continue
            
        ll = score_utterance(test_wav, speaker_model)
        accepted = ll > THRESHOLD
        
        futuristic_speaker_ui('verify', 'unlocked' if accepted else 'locked', ll, THRESHOLD)
        time.sleep(1)
        
except KeyboardInterrupt:
    print("System stopped.")

System stopped.


### 7️⃣ Multi‑Speaker Identification


Here we extend the system to distinguish **among several enrolled speakers**. First, you enroll multiple users. Then, in a live loop, the system tells you **who** is speaking.


In [ ]:
import librosa
import time

print("\n👥 Multi‑Speaker Identification Mode")
num_users = int(input("How many users to enroll? (at least 2): "))
all_models = {}

for i in range(num_users):
    name = input(f"Name of user {i+1}: ").strip()
    all_models[name] = enroll_speaker(name, duration_per_sample=4, num_samples=3)

def identify_speaker(wav, models_dict):
    scores = {}
    for name, (gmm, _) in models_dict.items():
        scores[name] = score_utterance(wav, gmm)
    best = max(scores, key=scores.get)
    return best, scores[best]

print("\n🔍 AI engine ready. Speak now. Ctrl+C to stop.\n")
time.sleep(1)

try:
    while True:
        test_wav = record_voice(duration=3, prompt="Speak...")
        test_wav, _ = librosa.effects.trim(test_wav, top_db=20)
        
        if len(test_wav) < RATE * 0.5:
            continue
            
        speaker, best_score = identify_speaker(test_wav, all_models)
        futuristic_speaker_ui('identify', speaker)
        time.sleep(1)
        
except KeyboardInterrupt:
    print("System stopped.")

🎤 Speak...
System stopped.



### 🏁 Conclusion



**What you accomplished in this project:**



1. **Voice Biometric Enrollment** – Captured natural speech and trained a Gaussian Mixture Model (GMM) to represent the unique vocal characteristics of a speaker.

2. **Threshold Calibration** – Built a practical method to decide “same speaker” vs. “different speaker” using enrolment data, and later strengthened it to reject silence and noise.

3. **Live Speaker Verification** – Ran a real‑time system that listens through the microphone and instantly grants or denies access based on who is speaking.

4. **Multi‑Speaker Identification** – Extended the system to recognise *which* enrolled user is talking, not just whether the voice belongs to the authorised person.



This is exactly how modern voice assistants, phone lock screens, and banking apps verify identity – entirely on‑device, with no cloud needed.



**You now have a working biometric ear inside your Python environment.**






**In the next video:**



**Project 6 – Intelligent Noise Removal with AI**  

  Use spectral gating and deep‑learning‑based denoising to clean noisy recordings without damaging the original voice – perfect for podcasts, meetings, and real‑world recordings. 
